<a id="intro"></a>
# ЛР № 5. Расчет метрик Airbnb Amsterdam

Это первая часть общего аналитического ноутбука ЛР № 5 и № 7.
Здесь читаем готовый `mart`, рассчитываем метрики и проверяем результаты.

Предыдущий этап: [витрина ЛР № 4](../docs/mart.md).  
Сквозной вопрос: [Project Brief](../docs/project_brief.md).  
Описания, определения и вывод: [docs/metrics.md](../docs/metrics.md).

## 1. Вход и параметры

In [1]:
# M01. Параметры
DATASET_SLUG = "airbnb_amsterdam_listings"
SOURCE_INGESTED_ON = "2026-10-23"
MART_PATH = f"s3://mart/{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}/listings.parquet"
MIN_OBSERVATIONS = 20

## 2. Подключение

In [2]:
# M02. Подключение
import os
import sys
from html import escape
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/common/s3.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
sys.path.insert(0, str(PROJECT_ROOT))
from src.common.s3 import configure_s3

connection = duckdb.connect()
settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing = [name for name in settings if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing)}")
configure_s3(connection)
print(f"Вход: {MART_PATH}")


def select_table(sql):
    cursor = connection.execute(sql)
    header = "".join(f"<th>{escape(column[0])}</th>" for column in cursor.description)
    rows = cursor.fetchall()
    body = "".join(
        "<tr>" + "".join(
            f"<td>{escape(str(value)) if value is not None else '—'}</td>" for value in row
        ) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    print(f"Строк результата: {len(rows)}")
    return rows

Вход: s3://mart/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/listings.parquet


## 3. Чтение витрины

Одна строка витрины — одно объявление. При чтении сохраняем все объявления снимка.

In [3]:
# M03. Вход
connection.read_parquet(MART_PATH, hive_partitioning=False).create_view("listings")

connection.execute("""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT id, district, room_type, last_scraped, calendar_last_scraped,
           CASE WHEN price_eur > 0
                     AND flag_price_missing IS FALSE
                     AND flag_price_invalid IS FALSE
                THEN price_eur END AS valid_price_eur,
           CASE WHEN has_availability IS TRUE AND availability_365 BETWEEN 0 AND 365
                THEN availability_365 END AS valid_available_days
    FROM listings
""")

input_rows = select_table("""
    SELECT count(*) AS records_count,
           count(DISTINCT id) AS unique_ids,
           count(DISTINCT district) AS districts_count,
           count(DISTINCT room_type) AS room_types_count,
           min(last_scraped) AS first_scrape,
           max(last_scraped) AS last_scrape,
           min(calendar_last_scraped) AS first_calendar_scrape,
           max(calendar_last_scraped) AS last_calendar_scrape
    FROM analysis_base
""")
total_rows, unique_ids = input_rows[0][:2]
assert total_rows > 0 and unique_ids == total_rows, "Ключ id должен быть заполнен и уникален"

records_count,unique_ids,districts_count,room_types_count,first_scrape,last_scrape,first_calendar_scrape,last_calendar_scrape
10369,10369,22,4,2026-06-15,2026-06-24,2026-06-15,2026-06-24


Строк результата: 1


<a id="definitions"></a>
## 4. Что считаем

Число объявлений, медианную заявленную цену и медианную календарную доступность.
Правила отбора значений указаны в [docs/metrics.md](../docs/metrics.md).

## 5. Общий итог

Считаем медианы по пригодным значениям, а число объявлений — по всему снимку.

In [4]:
# M04. Общие метрики
connection.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT *,
           100.0 * (listings_count - price_count) / nullif(listings_count, 0) AS unusable_price_percent
    FROM (
        SELECT
           count(DISTINCT id) AS listings_count,
           count(valid_price_eur) AS price_count,
           count(valid_available_days) AS calendar_count,
           median(valid_price_eur) AS median_price_eur,
           median(valid_available_days) AS median_available_days
        FROM analysis_base
    ) AS grouped
""")
overall_rows = select_table("""
    SELECT
           listings_count, price_count, calendar_count,
           round(median_price_eur, 2) AS median_price_eur,
           median_available_days,
           round(unusable_price_percent, 2) AS unusable_price_percent
    FROM overall_metrics
""")

listings_count,price_count,calendar_count,median_price_eur,median_available_days,unusable_price_percent
10369,6377,10081,291.00,22.0,38.5


Строк результата: 1


## 6. Сравнение по районам и окно

Одна строка результата — один район. Оконная функция добавляет долю района в общем числе объявлений.

In [5]:
# M05. По районам
connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE district_metrics AS
    WITH grouped AS (
        SELECT district,
           count(DISTINCT id) AS listings_count,
           count(valid_price_eur) AS price_count,
           count(valid_available_days) AS calendar_count,
           median(valid_price_eur) AS median_price_eur,
           median(valid_available_days) AS median_available_days
        FROM analysis_base
        GROUP BY district
    )
    SELECT *,
           100.0 * (listings_count - price_count) / nullif(listings_count, 0) AS unusable_price_percent,
           100.0 * listings_count / nullif(sum(listings_count) OVER (), 0) AS percent_of_listings,
           price_count < {MIN_OBSERVATIONS} AS small_price_group,
           calendar_count < {MIN_OBSERVATIONS} AS small_calendar_group
    FROM grouped
""")
district_rows = select_table("""
    SELECT district,
           listings_count, price_count, calendar_count,
           round(median_price_eur, 2) AS median_price_eur,
           median_available_days,
           round(unusable_price_percent, 2) AS unusable_price_percent,
           round(percent_of_listings, 3) AS percent_of_listings,
           small_price_group, small_calendar_group
    FROM district_metrics
    ORDER BY district
""")

district,listings_count,price_count,calendar_count,median_price_eur,median_available_days,unusable_price_percent,percent_of_listings,small_price_group,small_calendar_group
Bijlmer-Centrum,62,47,58,160.00,58.5,24.19,0.598,False,False
Bijlmer-Oost,39,24,38,120.30,18.5,38.46,0.376,False,False
Bos en Lommer,546,257,539,265.00,3.0,52.93,5.266,False,False
Buitenveldert - Zuidas,119,87,118,239.80,99.5,26.89,1.148,False,False
Centrum-Oost,921,666,890,287.79,95.0,27.69,8.882,False,False
Centrum-West,1191,851,1147,297.52,75.0,28.55,11.486,False,False
De Aker - Nieuw Sloten,71,46,69,209.17,55.0,35.21,0.685,False,False
De Baarsjes - Oud-West,1818,1066,1786,330.49,14.5,41.36,17.533,False,False
De Pijp - Rivierenbuurt,1195,696,1157,335.55,17.0,41.76,11.525,False,False
Gaasperdam - Driemond,54,33,54,162.20,49.0,38.89,0.521,False,False


Строк результата: 22


## 7. Уточнение и отдельные значения

Добавляем тип жилья к группировке по районам. Для будущего графика сохраняем отдельные цены.

In [6]:
# M06. Результаты для продолжения анализа
connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE segment_metrics AS
    WITH grouped AS (
        SELECT district, room_type,
           count(DISTINCT id) AS listings_count,
           count(valid_price_eur) AS price_count,
           count(valid_available_days) AS calendar_count,
           median(valid_price_eur) AS median_price_eur,
           median(valid_available_days) AS median_available_days
        FROM analysis_base
        GROUP BY district, room_type
    )
    SELECT *,
           100.0 * (listings_count - price_count) / nullif(listings_count, 0) AS unusable_price_percent,
           price_count < {MIN_OBSERVATIONS} AS small_price_group,
           calendar_count < {MIN_OBSERVATIONS} AS small_calendar_group
    FROM grouped
""")
print("Первые 10 строк группировки по району и типу жилья:")
segment_preview = select_table("""
    SELECT district, room_type,
           listings_count, price_count, calendar_count,
           round(median_price_eur, 2) AS median_price_eur,
           median_available_days,
           round(unusable_price_percent, 2) AS unusable_price_percent,
           small_price_group, small_calendar_group
    FROM segment_metrics
    ORDER BY district, room_type
    LIMIT 10
""")
segment_count = connection.execute("SELECT count(*) FROM segment_metrics").fetchone()[0]
print(f"Строк уточняющей группировки: {segment_count}")

print("Целое жилье в двух районах с наибольшим числом таких объявлений:")
comparison_rows = select_table("""
    SELECT district, room_type,
           listings_count, price_count, calendar_count,
           round(median_price_eur, 2) AS median_price_eur,
           median_available_days,
           round(unusable_price_percent, 2) AS unusable_price_percent,
           small_price_group, small_calendar_group
    FROM segment_metrics
    WHERE room_type = 'Entire home/apt'
    ORDER BY listings_count DESC, district
    LIMIT 2
""")

connection.execute("""
    CREATE OR REPLACE TEMP TABLE distribution_values AS
    SELECT district, room_type, valid_price_eur AS price_eur
    FROM analysis_base
    WHERE valid_price_eur IS NOT NULL
""")
price_summary = select_table("""
    SELECT count(*) AS price_count, min(price_eur) AS min_price_eur,
           median(price_eur) AS median_price_eur, max(price_eur) AS max_price_eur
    FROM distribution_values
""")[0]

Первые 10 строк группировки по району и типу жилья:


district,room_type,listings_count,price_count,calendar_count,median_price_eur,median_available_days,unusable_price_percent,small_price_group,small_calendar_group
Bijlmer-Centrum,Entire home/apt,26,15,26,188.33,6.5,42.31,True,False
Bijlmer-Centrum,Hotel room,10,10,10,227.20,364.0,0.0,True,True
Bijlmer-Centrum,Private room,26,22,22,118.48,187.0,15.38,False,False
Bijlmer-Oost,Entire home/apt,28,13,27,285.50,2.0,53.57,True,False
Bijlmer-Oost,Private room,11,11,11,104.00,81.0,0.0,True,True
Bos en Lommer,Entire home/apt,507,228,500,281.14,0.0,55.03,False,False
Bos en Lommer,Private room,39,29,39,126.00,58.0,25.64,False,False
Buitenveldert - Zuidas,Entire home/apt,87,56,86,298.58,41.5,35.63,False,False
Buitenveldert - Zuidas,Private room,32,31,32,148.50,246.5,3.13,False,False
Centrum-Oost,Entire home/apt,614,399,606,382.50,48.0,35.02,False,False


Строк результата: 10
Строк уточняющей группировки: 55
Целое жилье в двух районах с наибольшим числом таких объявлений:


district,room_type,listings_count,price_count,calendar_count,median_price_eur,median_available_days,unusable_price_percent,small_price_group,small_calendar_group
De Baarsjes - Oud-West,Entire home/apt,1633,901,1621,357.00,10.0,44.83,False,False
De Pijp - Rivierenbuurt,Entire home/apt,1057,576,1033,361.00,10.0,45.51,False,False


Строк результата: 2


price_count,min_price_eur,median_price_eur,max_price_eur
6377,14.70,291.00,11412.00


Строк результата: 1


<a id="validation"></a>
## 8. Проверка расчетов

Сверяем число объявлений и пригодных значений, сумму долей районов и размер выборки цен.

In [7]:
# M07. Проверки
check_rows = select_table("""
    WITH checks(check_no, check_name, actual, expected) AS (
        SELECT 1, 'Число объявлений по районам',
               (SELECT sum(listings_count) FROM district_metrics),
               (SELECT listings_count FROM overall_metrics)
        UNION ALL
        SELECT 2, 'Число объявлений по районам и типам жилья',
               (SELECT sum(listings_count) FROM segment_metrics),
               (SELECT listings_count FROM overall_metrics)
        UNION ALL
        SELECT 3, 'Число цен для медианы',
               (SELECT sum(price_count) FROM segment_metrics),
               (SELECT count(*) FROM listings
                WHERE price_eur > 0
                  AND flag_price_missing IS FALSE AND flag_price_invalid IS FALSE)
        UNION ALL
        SELECT 4, 'Число календарных значений для медианы',
               (SELECT sum(calendar_count) FROM segment_metrics),
               (SELECT count(*) FROM listings
                WHERE has_availability IS TRUE AND availability_365 BETWEEN 0 AND 365)
        UNION ALL
        SELECT 5, 'Сумма долей районов, %',
               (SELECT sum(percent_of_listings) FROM district_metrics), 100.0
        UNION ALL
        SELECT 6, 'Число отдельных цен в выборке',
               (SELECT count(*) FROM distribution_values),
               (SELECT price_count FROM overall_metrics)
    )
    SELECT check_name, round(actual, 4) AS actual, round(expected, 4) AS expected,
           abs(actual - expected) < 1e-8 AS passed
    FROM checks
    ORDER BY check_no
""")
assert all(row[3] for row in check_rows), "Проверки не сошлись: вернитесь к расчетам"
print(f"Проверок пройдено: {len(check_rows)} из {len(check_rows)}")

check_name,actual,expected,passed
Число объявлений по районам,10369.0,10369.0,True
Число объявлений по районам и типам жилья,10369.0,10369.0,True
Число цен для медианы,6377.0,6377.0,True
Число календарных значений для медианы,10081.0,10081.0,True
"Сумма долей районов, %",100.0,100.0,True
Число отдельных цен в выборке,6377.0,6377.0,True


Строк результата: 6
Проверок пройдено: 6 из 6


<a id="continuation"></a>
## Продолжение в ЛР № 7

Подготовлены временные таблицы `district_metrics`, `segment_metrics` и
`distribution_values`. Общий итог находится в `overall_metrics`.
Для продолжения выполните ячейки M01–M07 после перезапуска ядра.